# Analysis and Visualization of Complex Agro-Environmental Data
---
### Exercise #5 - correction

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import seaborn as sns # For plotting
import matplotlib.pyplot as plt # For showing plots
import scipy.stats as sts
import scikit_posthocs as sp
import statsmodels.stats as stm
from statsmodels.graphics.gofplots import qqplot
import math

In [ ]:
df = pd.read_csv('../Examples-local/EFIplus_medit.zip',compression='zip', sep=";")

In [ ]:
# clean up the dataset to remove unnecessary columns (eg. REG) 
df.drop(df.iloc[:,5:15], axis=1, inplace=True)

# let's rename some columns so that they make sense
df.rename(columns={'Sum of Run1_number_all':'Total_fish_individuals'}, inplace=True) # inplace="True" means that df will be updated

# for sake of consistency, let's also make all column labels of type string
df.columns = list(map(str, df.columns))

In [ ]:
# a good way of detecting missing values in the dataset
plt.figure(figsize=(12,4))
sns.heatmap(df.isnull(),cbar=False,cmap='viridis',yticklabels=False)
plt.title('Missing values (yellow) in the dataset');

In [ ]:
df = df.dropna() # drops rows when at least one element is a missing value

#### Exercise 5.1
Test whether the means (or medians) of “Mean Annual Temperature” between presence and absence sites of *Salmo trutta fario* (Brown Trout) are equal using an appropriate test. Use both standardized and non-standardized values and compare results. Please state which is/are the null hypothesis of your test(s).


Standardize the Mean Annual Temperature (temp_ann)

In [ ]:
# standardize the Mean Annual Temperature
df['temp_ann_st'] = (df['temp_ann'] - df['temp_ann'].mean()) / df['temp_ann'].std() # standardization formula: (x - mean) / std
df['temp_ann_st']

In [ ]:
# simpler alternative
sts.zscore(df['temp_ann']) # standardizes the variable and returns a new array with the standardized values. It does not modify the original dataframe.


View the histograms of both the original and the standardized variable

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True)

sns.histplot(df['temp_ann'], ax=axes[0])
sns.histplot(df['temp_ann_st'], color='red', ax=axes[1])

plt.show()

Boxplots

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(4, 6))

sns.boxplot(data=df,x='Salmo trutta fario',y='temp_ann', ax=axes[0])
sns.boxplot(data=df,x='Salmo trutta fario',y='temp_ann_st', color='red', ax=axes[1])

plt.show()

Run a t-test with non-standardized variable

In [ ]:
# Run t test
# H0 : The samples are drawn from populations with equal means

sample1 = df[df['Salmo trutta fario']==0]['temp_ann']
sample2 = df[df['Salmo trutta fario']==1]['temp_ann']

print('Mean of sample 1 = ', sample1.mean())
print('Mean of sample 2 = ', sample2.mean())

# t-test - tests the null hypothesis that sample 1 and 2 are derived from populations with the same mean
stat, p = sts.ttest_ind(sample1, sample2)
print('Statistics=%.3f, p=%.3f' % (stat, p)) # print outputs
alpha=0.05
if p > alpha:
 print('fail to reject H0. Rejecting H0 has an error probability >0.05')
else:
 print('reject H0 with an error probability <0.05)')

Run a t-test with the standardized variable

In [ ]:
# Run t test on standardized variables
# H0 : The samples are drawn from populations with equal means

sample1 = df[df['Salmo trutta fario']==0]['temp_ann_st']
sample2 = df[df['Salmo trutta fario']==1]['temp_ann_st']

print('Mean of sample 1 = ', sample1.mean())
print('Mean of sample 2 = ', sample2.mean())

# t-test - tests the null hypothesis that sample 1 and 2 are derived from populations with the same mean
stat, p = sts.ttest_ind(sample1, sample2)
print('Statistics=%.3f, p=%.3f' % (stat, p)) # print outputs
alpha=0.05
if p > alpha:
 print('fail to reject H0. Rejecting H0 has an error probability >0.05')
else:
 print('reject H0 with an error probability <0.05)')

The result of the t-test is the same for both the original and standardized variable, which is expected since standardization does not change the distribution of the data, it only rescales it.

#### Exercise 5.2
Test if the frequency of sites with presence and absence of *Salmo trutta fario* (Brown Trout) are independent from the country. Please state which is/are the null hypothesis of your test(s). You may try to produce an alluvial plot.

In [ ]:
# produce contingency table of Country and Samo trutta fario.
cdf = pd.crosstab(index=df['Country'], columns=df['Salmo trutta fario'])
print(cdf)

Run a Chi-square test of independency

In [ ]:
# Chi-square test of independency
stat, p, dof, expected = sts.chi2_contingency(cdf) # the function returns the chi-square statistic, the p-value, the degrees of freedom and the expected frequencies based on the observed data
print('df=%d' % dof) # print degrees of freedom
print('expected values:')
print(expected) # print the expected frequencies based on the observed data


The table above shows the values in the contingency table would be expected in case the two variables (country and trout presence/absence) were independent. The Chi-square test will check if the observed values are significally different from these expected values

In [ ]:

# Alternative 1: interpret based on test-statistic
prob=0.95
critical = sts.chi2.ppf(prob, dof) # the function returns the critical value of the chi-square distribution for a given probability and degrees of freedom. It is the value that the test statistic must exceed to reject the null hypothesis at the given significance level (1 - prob).
print('critical=%.3f, stat=%.3f' % (critical, stat)) # print the critical value and the test statistic
if abs(stat) >= critical: # if the absolute value of the test statistic is greater than or equal to the critical value, we reject the null hypothesis that the variables are independent
 print('stat > critical => reject H0 that variables are independent')
else: # if the absolute value of the test statistic is less than the critical value, we fail to reject the null hypothesis that the variables are independent
 print('stat < critical => fail to reject H0 that variables are independent')


In [ ]:
 # Alternative 2: interpret based on p-value
alpha = 0.05 # the significance level is the threshold for rejecting the null hypothesis. It represents the probability of rejecting the null hypothesis when it is actually true (Type I error). A common choice for alpha is 0.05, which means that we are willing to accept a 5% chance of making a Type I error.
print('significance=%.2f, p=%.3f' % (alpha, p)) # print the significance level and the p-value. The p-value is the probability of observing a test statistic as extreme as, or more extreme than, the one observed, under the null hypothesis. If the p-value is less than or equal to the significance level (alpha), we reject the null hypothesis that the variables are independent. If the p-value is greater than the significance level, we fail to reject the null hypothesis
if p <= alpha: # if the p-value is less than or equal to the significance level, we reject the null hypothesis that the variables are independent
 print('reject H0 that variables are independent')
else:
 print('fail to reject H0 that variables are independent')

Produce an Aluvial plot

In [ ]:
# Create a contingency table
cdf = pd.crosstab(index=df['Country'], columns=df['Salmo trutta fario']) 

# Convert contingency table to long format
cdf_long = cdf.reset_index().melt( # the function melt() is used to transform the contingency table from a wide format to a long format. The reset_index() method is called before melt() to convert the index of the contingency table (which is 'Country') into a regular column, so that it can be used as an identifier variable in the melt() function. The id_vars parameter specifies the column(s) to use as identifier variables (in this case, 'Country'), while var_name and value_name parameters specify the names of the new columns that will be created for the variable names and values, respectively. The resulting cdf_long dataframe will have three columns: 'Country', 'Occurrence' (which contains the values from the original columns), and 'Count' (which contains the corresponding counts).
    id_vars='Country',
    var_name='Occurrence',
    value_name='Count'
)
print(cdf_long)


In [ ]:
# Labels (nodes)
countries = ['Italy', 'Spain', 'Portugal']
occurrence = ['Absence', 'Presence']
labels = countries + occurrence

# Map labels to indices
label_to_index = {label: i for i, label in enumerate(labels)}

# Links
sources = cdf_long['Country'].map(label_to_index)
targets = cdf_long['Occurrence'].map({0: label_to_index['Absence'],
                                      1: label_to_index['Presence']})
values = cdf_long['Count']



In [ ]:
import plotly.graph_objects as go # the module graph_objects from the plotly library is imported and aliased as go. This module provides classes and functions for creating various types of plots and visualizations, including Sankey diagrams, which are used to visualize flow and relationships between different categories or nodes in a dataset. By importing this module, we can use its functionalities to create a Sankey diagram based on the contingency table we have created.

# Create Sankey diagram
fig = go.Figure(go.Sankey( 
    node=dict(# define nodes:
        label=labels, # the label parameter is set to the list of labels that we have defined earlier
        pad=15, # the pad parameter specifies the amount of padding (in pixels) between the nodes in the Sankey diagram.
        thickness=20 # the thickness parameter specifies the thickness of the nodes in the Sankey diagram. 
    ),
    link=dict(# define links:
        source=sources, # the source parameter is set to the list of source indices that we have created by mapping the 'Country' column of the cdf_long dataframe to the corresponding indices in the labels list.
        target=targets, # the target parameter is set to the list of target indices that we have created by mapping the 'Occurrence' column of the cdf_long dataframe to the corresponding indices in the labels list (0 for 'Absence' and 1 for 'Presence').
        value=values # the value parameter is set to the list of counts that we have extracted from the 'Count' column of the cdf_long dataframe. This parameter determines the width of the links in the Sankey diagram, with wider links representing higher counts.
    )
), layout=dict(width=700, height=600)) # the layout parameter is used to specify the overall layout of the figure, including its width and height in pixels.

fig.show()

#### Exercise 5.3
Test whether there are diferences in the mean elevation in the upstream catchment (Elevation_mean_catch) among the eight most sampled catchments (consider that mean elevation is normally distributed). For which pairs of catchments are these diferences significant? Please state which is/are the null hypothesis of your test(s).

Check the distribution of Mean elevtion in the upsream catchment

In [ ]:
sns.histplot(df['Elevation_mean_catch'])

In [ ]:
from statsmodels.graphics.gofplots import qqplot

qqplot(pd.Series(df['Elevation_mean_catch']), line='s')
plt.show()

Select the dataset for the eight most sampled catchments

In [ ]:
catchment_count = pd.crosstab(index = df['Catchment_name'], columns='count') # the function crosstab() is used to create a contingency table that counts the occurrences of each unique value in the 'Catchment_name' column. The index parameter specifies the column to use as the index of the resulting table, and the columns parameter specifies the name of the column that will contain the counts. The resulting table will have one row for each unique value in 'Catchment_name' and a single column named 'count' that contains the count of occurrences for each catchment.
catch_top8 = catchment_count.sort_values(by=['count'], ascending=False).head(8).index.to_list() # the function sort_values() is used to sort the contingency table by the 'count' column in descending order (ascending=False). The head(8) method is then used to select the top 8 rows of the sorted table, which correspond to the 8 catchments with the highest counts. Finally, the index of these rows is extracted and converted to a list using the to_list() method, resulting in a list of the names of the top 8 catchments.
dfsub = df[df.Catchment_name.isin(catch_top8)] # subset the original dataframe to include only the rows where the 'Catchment_name' column contains values that are in the catch_top8 list. The isin() method is used to check if each value in the 'Catchment_name' column is present in the catch_top8 list, and the resulting boolean mask is used to filter the dataframe. The resulting dfsub dataframe will contain only the rows corresponding to the top 8 catchments with the highest counts.

Explore data with bioxplots

In [ ]:
# Boxplots
plt.figure(figsize = (10,6))
sns.boxplot(data=dfsub, x='Catchment_name', y='Elevation_mean_catch')
plt.show()

Run ANOVA test

In [ ]:
# Although the distribution of the mean elevation is right skewed and seems to depart from normality we will nevertheless try to run ANOVA. 

mod = ols('Elevation_mean_catch ~ Catchment_name',# the formula for the ANOVA model. The dependent variable is Elevation_mean_catch and the independent variable is Catchment_name (categorical variable with 4 levels, one for each catchment). The formula is written in the format 'dependent_variable ~ independent_variable'.
            data=dfsub).fit() # fit the ANOVA model using ordinary least squares (OLS) regression. The function ols() from the statsmodels library is used to specify the model, and the fit() method is called to estimate the parameters of the model based on the data in dfsub.
                
aov_table = sm.stats.anova_lm(mod, typ=2) # typ is the type of anova type to perform ('I','II' or 'III' = 1,2,3); the choice of the type of ANOVA depends on the design of the experiment and the research question. Type I ANOVA (sequential) is appropriate when the order of the factors is important, while Type II ANOVA (hierarchical) is appropriate when the order of the factors is not important. Type III ANOVA (marginal) is appropriate when there are interactions between factors or when there are unbalanced designs.
print(aov_table) # provides the usual ANOVA table

alpha=0.05
p=aov_table['PR(>F)'].iloc[0] # the p-value is located in the column 'PR(>F)' and in the row corresponding to the factor of interest (Catchment_name). Since there is only one factor in this ANOVA model, we can use iloc[0] to access the first row of the 'PR(>F)' column, which corresponds to the p-value for the effect of Catchment_name on Elevation_mean_catch.

if p <= alpha: # if the p-value is less than or equal to the significance level, we reject the null hypothesis that the mean elevation values are equal among catchments
 print('reject H0 that mean elevation values are equal among catchments')
else:
 print('fail to reject H0 that mean elevation values are equal among catchments')

# compute mean elevation for eacch catchment
dfsub[['Elevation_mean_catch','Catchment_name']].groupby('Catchment_name').mean()


Run multiple comparisons post-hoc test (Tukey's HSD) to determine which catchments differ in mean elevation

In [ ]:
# Multiple comparisons - perform Tukey's test 
tukey = stm.multicomp.pairwise_tukeyhsd(endog=dfsub['Elevation_mean_catch'],
                          groups=dfsub['Catchment_name'],
                          alpha=0.05)
#display results
print(tukey)

#### Exercise 5.4
Run the non-parametric equivalent of the test you used in the previous exercise and compare with the ANOVA test


In [ ]:
# Compute median values of Mean elevation for each catchment
dfsub[['Elevation_mean_catch','Catchment_name']].groupby('Catchment_name').median()

In [ ]:
# Run non-parametric equivalent to one-way ANOVA - Kruskal-Walis test

# ALTERNATIVE 1: long code with one line per sample

sample1 = df[(df['Catchment_name']=='Cantabrica')]['Elevation_mean_catch']
sample2 = df[(df['Catchment_name']=='Douro')]['Elevation_mean_catch']
sample3 = df[(df['Catchment_name']=='Galiza-Norte')]['Elevation_mean_catch']
sample4 = df[(df['Catchment_name']=='Galiza-Sul')]['Elevation_mean_catch']
sample5 = df[(df['Catchment_name']=='Guadia')]['Elevation_mean_catch']
sample6 = df[(df['Catchment_name']=='Minho')]['Elevation_mean_catch']
sample7 = df[(df['Catchment_name']=='Mondego')]['Elevation_mean_catch']
sample8 = df[(df['Catchment_name']=='Tejo')]['Elevation_mean_catch']


stat, p = sts.kruskal(sample1, sample2, sample3, sample4, sample5, sample6, sample7, sample8)
print('F-statistics=%.3f, p=%.6f' % (stat, p))

alpha=0.05

if p <= alpha:
 print('reject H0 that median elevation values are equal among catchments')
else:
 print('fail to reject H0 that median elevation values are equal among catchments')


In [ ]:
# ALTERNATIVE 2 - more compact code using dictionary comprehension

# The code creates a dictionary called samples where the keys are the unique values of the 'Catchment_name' column and the values are the corresponding 'Elevation_mean_catch' values for each catchment. The groupby() method is used to group the dataframe by 'Catchment_name', and the dictionary comprehension iterates over each group (c, g) to extract the 'Elevation_mean_catch' values for that catchment and store them in the dictionary.
samples = {
    c: g['Elevation_mean_catch'] # for each catchment (c) and its corresponding group of rows (g) in the dataframe, we extract the 'Elevation_mean_catch' values and store them in the dictionary with the catchment name as the key.
    for c, g in dfsub.groupby('Catchment_name') # the groupby() method is used to group the dataframe by 'Catchment_name', and the dictionary comprehension iterates over each group (c, g) to extract the 'Elevation_mean_catch' values for that catchment and store them in the dictionary with the catchment name as the key.
}

stat, p = sts.kruskal(*samples.values()) # the function kruskal() from the scipy.stats library is used to perform the Kruskal-Wallis H-test, which is a non-parametric test for comparing the medians of two or more independent samples. The * operator is used to unpack the values of the samples dictionary, which are the 'Elevation_mean_catch' values for each catchment, and pass them as separate arguments to the kruskal() function.
print('F-statistics=%.3f, p=%.6f' % (stat, p)) # print outputs

alpha=0.05

if p <= alpha:
 print('reject H0 that median elevation values are equal among catchments')
else:
 print('fail to reject H0 that median elevation values are equal among catchments')

Run multiple comparisons *a posteriori* tests 

In [ ]:
samples = dfsub.groupby('Catchment_name')['Elevation_mean_catch']

In [ ]:
# Create list of samples - alternative 1: long code with one line per sample
samples = [sample1, sample2, sample3, sample4, sample5, sample6, sample7, sample8]

# Create list of samples - alternative 2: more compact code using dictionary comprehension
samples = [group.values for _, group in dfsub.groupby('Catchment_name')['Elevation_mean_catch']] # the code creates a list of samples by iterating over each group of 'Elevation_mean_catch' values for each catchment in the dfsub dataframe. The groupby() method is used to group the dataframe by 'Catchment_name', and the list comprehension iterates over each group (represented by the underscore _ for the catchment name and group for the corresponding 'Elevation_mean_catch' values) to extract the values and store them in a list. The resulting samples list will contain the 'Elevation_mean_catch' values for each catchment as separate elements.

sp.posthoc_conover(samples, p_adjust = 'bonferroni') # the correction for multiple comparisons is based on the bonferroni's correction.
# the output is a matrix of p-values for each pair of groups